### Dataset Information
This cell contains the file paths for the diagnostic and clinical notes datasets.


In [ ]:
 # datasset path od diagonos - /teamspace/uploads/projects/01kn719x6xtsmsyfy0rmv0d7zq/Uploads/DIAGNOSES_ICD.csv

### Dataset Information
This cell contains the file paths for the diagnostic and clinical notes datasets.


In [ ]:
# dataset of notes - /teamspace/uploads/NOTEEVENTS.csv

### Imports & Setup
Importing standard data science and deep learning libraries like pandas, PyTorch, and Transformers.


In [ ]:
print('hello')
import pandas as pd, os, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
print('hello')


### Data Loading & Preprocessing
This section sets up paths, loads MIMIC-III clinical notes (`NOTEEVENTS.csv`), and filters specifically for `Discharge summary` texts as they hold the most valuable comprehensive clinical info.

Next, it loads `DIAGNOSES_ICD.csv`, defines a helper function `is_cancer()` to parse ICD-9 codes and classify if a diagnosis falls under cancer (ICD-9 codes 140-239), and finally merges the texts and labels together into a single dataframe.


In [ ]:
# ── 1. CONFIG & DATA LOADING ──────────────────────────

BASE = ""

# Load clinical notes
notes_df = pd.read_csv(BASE + "NOTEEVENTS.csv",
                        usecols=["SUBJECT_ID", "HADM_ID", "CATEGORY", "TEXT"])

# Filter for Discharge Summaries (most useful text)
notes_df = notes_df[notes_df["CATEGORY"] == "Discharge summary"]
notes_df = notes_df.dropna(subset=["TEXT"])

# Load diagnoses to create labels
diag_df = pd.read_csv(BASE + "DIAGNOSES_ICD.csv",
                       usecols=["HADM_ID", "ICD9_CODE"])

# Define Cancer (ICD-9 codes 140-239)
def is_cancer(code):
    try:
        c = str(code).strip()
        if c[0].upper() in ["V", "E"]: return 0
        num = float(c[:3])
        return 1 if 140 <= num <= 239 else 0
    except: return 0

diag_df["cancer"] = diag_df["ICD9_CODE"].apply(is_cancer)
cancer_labels = diag_df.groupby("HADM_ID")["cancer"].max().reset_index()
cancer_labels.columns = ["HADM_ID", "label"]

# Merge Text with Labels
merged = notes_df.merge(cancer_labels, on="HADM_ID", how="inner")
merged = merged.drop_duplicates(subset="HADM_ID")
print(f"Final dataset size: {len(merged)} | Cancer cases: {merged['label'].sum()}")



### PyTorch Dataset Class (`MIMICNoteDataset`)
Here, we define a standard PyTorch `Dataset` that handles the tokenization of the raw clinical notes using the HuggingFace `BertTokenizer`. It ensures all sequences are padded or truncated to a uniform `max_length=512`.


In [ ]:
# ── 2. DATASET CLASS ──────────────────────────────────

class MIMICNoteDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }



### Medical BERT Architecture
We construct `BertCancerClassifier` which uses `emilyalsentzer/Bio_ClinicalBERT` as its backbone. `Bio_ClinicalBERT` is specifically pre-trained on MIMIC-III text, meaning it already possesses deep knowledge of clinical jargon. We attach a custom linear classification head with Dropout layers to prevent overfitting.


In [ ]:
# ── 3. MEDICAL BERT MODEL ─────────────────────────────

class BertCancerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Use Bio_ClinicalBERT for medical knowledge
        self.bert = BertModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token representation
        return self.classifier(cls)



### Training & Validation Loop
We perform a train/test split on our merged dataframe and initialize the `DataLoader`s.

The training loop runs for 3 epochs utilizing `torch.cuda.amp.GradScaler` for mixed-precision training. This significantly speeds up training and reduces VRAM usage. At the end of each epoch, validation accuracy is measured, and ultimately, the trained weights are saved to `clinical_bert_cancer.pth`.


In [ ]:
# ── 4. SETUP & TRAINING ───────────────────────────────

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# Split data
train_df, val_df = train_test_split(merged, test_size=0.2, stratify=merged["label"], random_state=42)

train_ds = MIMICNoteDataset(train_df["TEXT"].tolist(), train_df["label"].tolist(), tokenizer)
val_ds   = MIMICNoteDataset(val_df["TEXT"].tolist(), val_df["label"].tolist(), tokenizer)

# Use batch_size 8 to avoid "Out of Memory" errors on Kaggle P100/T4 GPUs
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False)

model     = BertCancerClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler() # Faster training

print("\nStarting Training...")
for epoch in range(3): # 3 Epochs is usually enough
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(ids, mask)
            loss   = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    # Validation loop
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_loader:
            ids, mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["label"].to(device)
            logits = model(ids, mask)
            preds  = logits.argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {100*correct/total:.2f}%")

# Save the brain for fusion later
torch.save(model.state_dict(), "clinical_bert_cancer.pth")
print("Model saved!")


### Test Print
A simple test print cell.


In [ ]:
print("hello")